# Sommelier — Vietnamese UV Isolated Venv Test on Google Colab

Runs the sommelier podcast pipeline inside a **clean isolated `venv`** (`/content/sommelier_env`) using **`uv`** to install minimal dependencies and automatically pulls audio clips from YouTube via `scripts/pick_clips.py`.


## 1. Sanity check the runtime

In [ ]:
!nvidia-smi -L
!python --version
!df -h /content | tail -1
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'avail', torch.cuda.is_available())

## 2. Clone the repo

If you're iterating on the `--lang vi` integration, swap in your fork's URL.

In [ ]:
%cd /content
![ -d sommelier ] || git clone https://github.com/tuanad121/sommelier.git
%cd /content/sommelier/podcast-pipeline

## 3. Install dependencies (~15 min)

Mirrors the README's three-step install order (torch first, then requirements, then re-pin torch). Adds the `chunkformer` package on top for the VN MoE third slot.

In [ ]:
# 1. Install uv package manager
!pip install -q uv yt-dlp

# 2. Create clean isolated virtual environment on Colab
!uv venv --allow-existing /content/sommelier_env

# 3. Define proposed optimized dependencies list (resolved & conflict-free)
PROPOSED_REQUIREMENTS = """
numpy==1.26.4
torch==2.7.1
torchaudio==2.7.1
torchvision==0.22.1
lightning==2.4.0
torchmetrics==1.7.4
onnxruntime-gpu
nemo-toolkit[asr]==2.5.3
pyannote.audio==3.3.2
speechbrain==1.0.3
faster-whisper==1.1.1
whisperx==3.3.1
chunkformer
ctranslate2>=4.0.0,<5.0.0
demucs>=4.0.0
panns-inference
librosa==0.11.0
soundfile==0.13.1
pydub==0.25.1
julius==0.2.7
numba==0.61.2
transformers==4.53.0
huggingface-hub==0.33.4
g2pk
jamo
nltk==3.9.1
openai==1.97.1
tritony==0.0.20
tritonclient[all]
pandas==2.3.1
PyYAML==6.0.2
tqdm==4.67.1
wandb==0.21.0
requests==2.32.4
einops==0.8.1
hydra-core==1.3.2
omegaconf==2.3.0
yt-dlp
"""

with open('requirements_proposed.txt', 'w') as f:
    f.write(PROPOSED_REQUIREMENTS.strip())

print('>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside venv...')
!uv pip install --python /content/sommelier_env torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

print('\n>>> Step 2: Testing resolution & installing proposed dependencies into isolated venv via UV (Realtime Log)...')
!uv pip install --python /content/sommelier_env -r requirements_proposed.txt --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match


In [ ]:
# Verify imports inside isolated venv
!/content/sommelier_env/bin/python -c "import torch, whisperx, demucs, nemo, pyannote.audio, transformers; from chunkformer import ChunkFormerModel; print('vEnv Python:', torch.__version__, 'CUDA:', torch.cuda.is_available()); print('whisperx:', whisperx.__version__); print('transformers:', transformers.__version__); print('nemo:', nemo.__version__)"


## 4. Hugging Face authentication

Pyannote diarization and Sortformer are gated. Accept the license on each model page first, then paste your token below.

In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass('HF token (hf_...): ')
login(token=hf_token)

# Also write it to config.json so pipeline code that reads cfg['huggingface_token'] works.
import json, pathlib
cfg_path = pathlib.Path('/content/sommelier/podcast-pipeline/config.json')
cfg = json.loads(cfg_path.read_text())
cfg['huggingface_token'] = hf_token
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
print('config.json updated')

## 5. Provide a Vietnamese audio clip

Upload one short (~30–60 s) WAV/MP3. The pipeline expects a **folder** of audio, so we drop the file into `/content/vi_audio/`.

In [ ]:
# Automated YouTube Audio Puller using scripts/pick_clips.py
import os, random, glob, shutil
from pathlib import Path

os.makedirs('/content/vi_audio', exist_ok=True)

# Create TSV file with target YouTube URL
yt_url = 'https://www.youtube.com/watch?v=ZzwZaHNqrVM'
tsv_path = '/content/yt_clip.tsv'
with open(tsv_path, 'w', encoding='utf-8') as f:
    f.write(f'yt_sample\t{yt_url}\t00:00\t03:00\tchannel_test\tpodcast\n')

print('>>> Pulling YouTube audio clip via pick_clips.py...')
!/content/sommelier_env/bin/python /content/sommelier/scripts/pick_clips.py --tsv /content/yt_clip.tsv --out /content/vi_audio

# Pick a random WAV file in /content/vi_audio for the pipeline test
wav_files = glob.glob('/content/vi_audio/*.wav')
if wav_files:
    selected_wav = random.choice(wav_files)
    print(f'\n✅ Successfully pulled YouTube audio! Selected for pipeline: {selected_wav}')
else:
    print('❌ Warning: No WAV file produced by pick_clips.py. Please check yt-dlp output.')


## 6. Run the pipeline (VN MoE, no Demucs/SepReformer for speed)

Flags below disable Demucs, SepReformer, Qwen3-Omni captioning, and the LLM post-pass to keep the smoke test fast and minimize VRAM. Add them back once the basic VN flow works.

In [ ]:
%cd /content/sommelier/podcast-pipeline
!/content/sommelier_env/bin/python main_original_ASR_MoE.py \
  --input_folder_path /content/vi_audio \
  --lang vi \
  --vad \
  --dia3 \
  --ASRMoE \
  --no-demucs \
  --whisperx_word_timestamps \
  --no-qwen3omni \
  --no-sepreformer \
  --LLM case_0 \
  --seg_th 0.11 \
  --min_cluster_size 11 \
  --clust_th 0.5 \
  --merge_gap 2


## 7. Inspect output

The script writes per-clip outputs under `/content/vi_audio/_final/_processed_llm-twelve-cases-*/<audio_name>/`.

In [ ]:
import glob, json, pathlib

json_paths = sorted(glob.glob('/content/vi_audio/_final/**/*.json', recursive=True))
print(f'Found {len(json_paths)} result file(s):')
for p in json_paths:
    print(' -', p)

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    print('\nMetadata:')
    print(json.dumps(result.get('metadata', {}), indent=2, ensure_ascii=False))
    print(f"\nFirst 3 segments of {len(result.get('segments', []))}:")
    for seg in result.get('segments', [])[:3]:
        print(json.dumps(seg, indent=2, ensure_ascii=False))


## Troubleshooting

**T4 OOM during MoE:** the three ASR models (Whisper-large-v3, PhoWhisper-large, ChunkFormer-CTC) plus Sortformer/pyannote barely fit in 16 GB. Either upgrade the runtime, or as a stopgap edit `asr_MoE()` in `main_original_ASR_MoE.py` to use `ThreadPoolExecutor(max_workers=1)` so models run sequentially instead of in parallel.

**`chunkformer` import error:** the package was added in step 3.4 above — if you skipped it, run `!pip install chunkformer` and **restart the runtime** (Runtime → Restart runtime).

**HF 401 on Sortformer / pyannote:** accept the license on each gated model page (`pyannote/segmentation-3.0`, `pyannote/embedding`, `nvidia/diar_sortformer_4spk-v1`) while logged in to the same HF account whose token you pasted.

**Empty `text_phowhisper` / `text_chunkformer` but `text_whisper` populated:** check the cell output for `PhoWhisper failed:` / `ChunkFormer failed:` log lines — the MoE swallows per-model errors so Whisper alone still produces a transcript.